# Día 1 — Del péndulo simple al caos determinista

### Taller: Física no lineal en el aula
**Congreso de Profesores de Física — Educación Secundaria**

---

## Cómo usar este cuaderno (leer antes de empezar)

Esto es un **cuaderno de Google Colab**. El código corre en una computadora de
Google, no en la suya: no hay que instalar nada.

**Los cuatro pasos del arranque:**

1. Arriba a la derecha, apretar **"Copiar en Drive"** (o *File → Save a copy in Drive*).
   👉 Si no hacen esto, pueden mirar pero **no guardar sus cambios**.
2. En el menú: **Entorno de ejecución → Ejecutar todas** (*Runtime → Run all*).
3. Si aparece un cartel que dice *"Este cuaderno no lo creó Google"*, apretar
   **"Ejecutar de todos modos"**.
4. Esperar ~20 segundos. Listo.

**Cómo se lee una celda:** cada bloque gris es una celda de código. A la izquierda
tiene un ▶. Si aparece `[ ]` no se ejecutó; si aparece `[3]` ya se ejecutó.

> ⚠️ **La regla de oro de Colab:** las celdas se ejecutan **en orden, de arriba
> hacia abajo**. Si algo da error raro, casi siempre es porque se salteó una celda.
> La solución universal: *Entorno de ejecución → Reiniciar y ejecutar todo*.

> 💡 **Sobre los deslizadores:** están configurados para recalcular **al soltar**,
> no mientras se arrastra. Muevan el deslizador hasta donde quieran y suelten;
> el gráfico se rehace en una fracción de segundo.

**Qué van a tener que hacer ustedes:**

- ✏️ **Para probar** → mover un deslizador y observar. No se escribe nada.
- 🧩 **Ejercicio** → cambiar **un número** en una celda y volver a ejecutarla (Shift+Enter).

No hace falta saber programar. Si algo no se entiende, se puede ignorar el código
y quedarse con los gráficos.

---
## 0. Preparación

Ejecutar esta celda una sola vez. No hay nada que instalar: Colab ya trae todo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk
from ipywidgets import interact, FloatSlider, IntSlider, SelectionSlider, Button, Output
from IPython.display import display

plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

g = 9.81   # aceleración de la gravedad, m/s²
print("Todo listo ✓   numpy", np.__version__)

---
## 1. El péndulo con aproximación de ángulo pequeño

La ecuación exacta del péndulo es

$$\ddot{\theta} = -\frac{g}{L}\,\sin\theta$$

y es **no lineal** por culpa del $\sin\theta$. El truco de siempre: si $\theta$ es
chico, $\sin\theta \approx \theta$, y queda

$$\ddot{\theta} = -\omega_0^2\,\theta, \qquad \omega_0 = \sqrt{g/L}$$

que sabemos resolver a mano:

$$\theta(t) = \theta_0\cos(\omega_0 t), \qquad T_0 = 2\pi\sqrt{L/g}$$

Lo notable de esta solución: **el período no depende de la amplitud** (isocronía).
Eso es lo que hace que un reloj de péndulo funcione... y es *falso*, como vamos a
ver en dos minutos.

In [ ]:
@interact(L=FloatSlider(min=0.1, max=2.0, step=0.05, value=1.0,
                        description="L (m)", continuous_update=False),
          theta0_grados=IntSlider(min=1, max=170, value=15,
                        description="θ₀ (°)", continuous_update=False))
def pendulo_lineal(L, theta0_grados):
    th0 = np.radians(theta0_grados)
    w0  = np.sqrt(g/L)
    T0  = 2*np.pi/w0
    t   = np.linspace(0, 4*T0, 1000)
    plt.figure()
    plt.plot(t, th0*np.cos(w0*t), lw=2)
    plt.axhline(0, color="gray", lw=0.6)
    plt.xlabel("t (s)"); plt.ylabel("θ (rad)")
    plt.title(f"Solución lineal:  θ(t) = θ₀·cos(ω₀t)      T₀ = {T0:.3f} s")
    plt.show()

✏️ **Para probar:** mover el ángulo inicial de 1° a 170°. La forma de la curva no
cambia y el período tampoco. Guarden esa observación: la vamos a romper enseguida.

---
## 2. Resolver la ecuación exacta: un integrador en 8 líneas

Como no queremos aproximar, hay que integrar numéricamente. Usamos
**Runge–Kutta de orden 4 (RK4)**, que es lo que hay adentro de casi cualquier
simulador. La idea: en vez de avanzar con la pendiente del punto inicial (que
sería el método de Euler), se promedian cuatro estimaciones de la pendiente
dentro del paso.

No hace falta entenderlo en detalle para usarlo, pero está bueno que lo vean:
son ocho renglones y no hay magia adentro.

In [ ]:
def rk4(f, u0, tmax, dt):
    "Integra UNA trayectoria. u0 = [theta, omega]."
    n  = int(tmax/dt)
    ts = np.linspace(0, n*dt, n+1)
    us = np.zeros((n+1, len(u0)))
    us[0] = u0
    for i in range(n):
        u, t = us[i], ts[i]
        k1 = f(u,             t)
        k2 = f(u + dt/2*k1,   t + dt/2)
        k3 = f(u + dt/2*k2,   t + dt/2)
        k4 = f(u + dt*k3,     t + dt)
        us[i+1] = u + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    return ts, us

def rk4_varias(U0, tmax, dt, guardar_cada=1, **kw):
    # Integra MUCHAS trayectorias a la vez. U0 tiene forma (2, N).
    # Es el mismo RK4, pero operando sobre vectores: numpy hace las N
    # trayectorias de una sola vez en vez de una por una. Para el grafico
    # del espacio de fases esto es ~9 veces mas rapido, y es lo que hace
    # que los deslizadores respondan sin demora.
    U, t = U0.copy(), 0.0
    guardadas = [U.copy()]
    for i in range(int(tmax/dt)):
        k1 = pendulo_vec(U,             t,          **kw)
        k2 = pendulo_vec(U + dt/2*k1,   t + dt/2,   **kw)
        k3 = pendulo_vec(U + dt/2*k2,   t + dt/2,   **kw)
        k4 = pendulo_vec(U + dt*k3,     t + dt,     **kw)
        U = U + dt/6*(k1 + 2*k2 + 2*k3 + k4)
        t += dt
        if (i+1) % guardar_cada == 0:
            guardadas.append(U.copy())
    return np.array(guardadas)

El estado del péndulo es $u = (\theta, \omega)$. La ecuación de segundo orden se
escribe como dos de primer orden:

$$\dot{\theta} = \omega, \qquad
\dot{\omega} = -\frac{g}{L}\sin\theta \;-\; b\,\omega \;+\; A\cos(\Omega t)$$

Por ahora dejamos $b = 0$ (sin rozamiento) y $A = 0$ (sin forzado). Los vamos a
encender en la sección 5.

In [ ]:
def pendulo(u, t, L=1.0, b=0.0, A=0.0, Omega=2.0):
    theta, omega = u
    return np.array([omega,
                     -(g/L)*np.sin(theta) - b*omega + A*np.cos(Omega*t)])

def pendulo_vec(U, t, L=1.0, b=0.0, A=0.0, Omega=2.0):
    "Igual, pero U tiene forma (2, N): N pendulos a la vez."
    return np.stack([U[1],
                     -(g/L)*np.sin(U[0]) - b*U[1] + A*np.cos(Omega*t)])

---
## 3. ¿Cuándo se rompe la aproximación?

Superponemos la solución lineal (línea punteada) con la solución numérica exacta
(línea llena).

In [ ]:
@interact(theta0_grados=IntSlider(min=1, max=179, value=20,
                        description="θ₀ (°)", continuous_update=False),
          L=FloatSlider(min=0.1, max=2.0, step=0.05, value=1.0,
                        description="L (m)", continuous_update=False))
def comparacion(theta0_grados, L):
    th0 = np.radians(theta0_grados)
    w0  = np.sqrt(g/L)
    T0  = 2*np.pi/w0
    ts, us = rk4(lambda u, t: pendulo(u, t, L=L), [th0, 0.0], 5*T0, 0.005)
    plt.figure()
    plt.plot(ts, us[:, 0], lw=2.2, label="exacto (numérico)")
    plt.plot(ts, th0*np.cos(w0*ts), lw=2, ls="--", label="aproximación lineal")
    plt.axhline(0, color="gray", lw=0.6)
    plt.xlabel("t (s)"); plt.ylabel("θ (rad)")
    plt.title(f"θ₀ = {theta0_grados}°")
    plt.legend(loc="lower left")
    plt.show()

✏️ **Para probar:** empezar en 10° (se superponen, no se distinguen), pasar a 45°
(se empiezan a desfasar) y llegar a 170° (no tienen nada que ver).

Fíjense en algo importante: el error **no** es que la solución exacta sea "fea".
Sigue siendo periódica y perfectamente ordenada. Lo que falla es que
**el período depende de la amplitud**.

---
## 4. El período exacto: existe, pero no es elemental

Integrando la conservación de la energía se llega a

$$T(\theta_0) = 4\sqrt{\frac{L}{g}}\; K\!\left(\sin^2\frac{\theta_0}{2}\right)$$

donde $K$ es la **integral elíptica completa de primera especie**. O sea: *sí* hay
fórmula cerrada, pero involucra una función que no es elemental y que hay que
evaluar numéricamente igual.

Este es un buen momento para el mensaje de fondo del taller: la frontera relevante
no es *"tiene fórmula / no tiene fórmula"*, sino **qué preguntas puedo responder
con lo que tengo**.

Abajo comparamos tres cosas: la predicción lineal (que dice que $T$ es constante),
la fórmula elíptica, y el período **medido sobre la simulación** — detectando
cuándo el péndulo vuelve a pasar por donde arrancó.

In [ ]:
def T_exacto(theta0, L=1.0):
    "Período exacto vía integral elíptica. Ojo: scipy usa el parámetro m = k²."
    return 4*np.sqrt(L/g)*ellipk(np.sin(theta0/2)**2)

def periodo_medido(theta0, L=1.0, dt=1e-3):
    "Mide el período buscando dos pasajes consecutivos por ω = 0 subiendo."
    ts, us = rk4(lambda u, t: pendulo(u, t, L=L),
                 [theta0, 0.0], 6*2*np.pi*np.sqrt(L/g), dt)
    w = us[:, 1]
    cruces = []
    for i in range(1, len(ts)):
        if w[i-1] < 0 <= w[i]:                      # cambio de signo − → +
            a = -w[i-1]/(w[i] - w[i-1])             # interpolación lineal
            cruces.append(ts[i-1] + a*dt)
            if len(cruces) == 2:
                break
    return cruces[1] - cruces[0] if len(cruces) == 2 else np.nan

In [ ]:
T0 = 2*np.pi*np.sqrt(1.0/g)

th_curva = np.radians(np.arange(1, 176, 2))
th_pts   = np.radians(np.arange(10, 171, 20))

plt.figure()
plt.plot(np.degrees(th_curva), [T_exacto(t)/T0 for t in th_curva],
         lw=2.5, label="fórmula elíptica")
plt.scatter(np.degrees(th_pts), [periodo_medido(t)/T0 for t in th_pts],
            s=55, zorder=5, color="crimson", label="medido en la simulación")
plt.axhline(1, ls="--", color="gray", label="predicción lineal")
plt.xlabel("θ₀ (grados)"); plt.ylabel("T(θ₀) / T₀")
plt.title("El período sí depende de la amplitud")
plt.legend()
plt.show()

Los puntos rojos caen exactamente sobre la curva: la fórmula elíptica y la
simulación coinciden (en las pruebas, con error relativo del orden de $10^{-13}$).
La recta gris es lo que predice la aproximación lineal, y se despega cada vez más.

🧩 **Ejercicio 1.** ¿A partir de qué amplitud el error de la aproximación lineal
supera el **1 %**?

Cambien el número de la celda de abajo y vuelvan a ejecutarla (Shift+Enter) hasta
que el resultado dé 1.0 %.

In [ ]:
theta_prueba = 20      # ←←← CAMBIAR ESTE NÚMERO (en grados)

# ---- no hace falta tocar nada de acá para abajo ----
err = 100*(T_exacto(np.radians(theta_prueba))/T0 - 1)
print(f"Con θ₀ = {theta_prueba}°  el período real es un {err:.2f} % más largo")
print(f"que el que predice la fórmula lineal.")

> **Para el aula:** este número explica por qué los relojes de péndulo usan
> amplitudes chicas, y por qué en el laboratorio de secundaria conviene medir
> $T$ con $\theta_0 < 15°$ si se quiere verificar $T = 2\pi\sqrt{L/g}$.

---
## 5. El espacio de fases

En vez de graficar $\theta$ contra $t$, graficamos $\omega$ contra $\theta$. Cada
estado del sistema es **un punto**; su evolución es **una curva**. Todo el
comportamiento posible del sistema entra en un solo dibujo.

Las flechas grises son el **campo vectorial**: en cada punto indican hacia dónde
se mueve el sistema. Las trayectorias simplemente siguen las flechas.

Ahora encendemos las dos cosas que rompen la conservación de la energía:

- **$b$** — rozamiento (disipación),
- **$A$** — forzado externo $A\cos(\Omega t)$ (inyección de energía).

In [ ]:
@interact(b=FloatSlider(min=0.0, max=1.0, step=0.02, value=0.0,
                        description="b (roce)", continuous_update=False),
          A=FloatSlider(min=0.0, max=2.5, step=0.05, value=0.0,
                        description="A (forzado)", continuous_update=False),
          Omega=FloatSlider(min=0.2, max=4.0, step=0.05, value=2.0,
                        description="Ω", continuous_update=False),
          tmax=IntSlider(min=10, max=150, step=10, value=40,
                        description="t max (s)", continuous_update=False))
def espacio_de_fases(b, A, Omega, tmax):
    fig, ax = plt.subplots(figsize=(7.5, 5))

    # --- campo vectorial (dibujado sin el forzado, que depende de t) ---
    th = np.linspace(-3*np.pi, 3*np.pi, 21)
    om = np.linspace(-8, 8, 15)
    TH, OM = np.meshgrid(th, om)
    dTH = OM
    dOM = -g*np.sin(TH) - b*OM
    n = np.hypot(dTH, dOM) + 1e-9
    ax.quiver(TH, OM, dTH/n, dOM/n, color="gray", alpha=0.45, width=0.0022)

    # --- trayectorias: las 6 a la vez, en un solo array (2, 6) ---
    CI = np.array([[-2.5, 0.5, 2.0, 0.0, 0.0, 0.0],
                   [ 0.0, 0.0, 0.0, 4.0, 6.5, -5.5]])
    T = rk4_varias(CI, float(tmax), 0.03, guardar_cada=2,
                   L=1.0, b=b, A=A, Omega=Omega)
    for j in range(CI.shape[1]):
        ax.plot(T[:, 0, j], T[:, 1, j], lw=1.4)
    ax.plot(CI[0], CI[1], "ko", ms=4)

    ax.set_xlim(-3*np.pi, 3*np.pi); ax.set_ylim(-8, 8)
    ax.set_xlabel("θ (rad)"); ax.set_ylabel("ω (rad/s)")
    ax.set_title(f"Espacio de fases     b = {b:.2f}    A = {A:.2f}")
    plt.show()

✏️ **Para probar, en este orden:**

1. **$b = 0$, $A = 0$.** Curvas cerradas (oscilación) y curvas abiertas arriba y
   abajo (el péndulo da vueltas enteras). La curva que separa ambos regímenes es
   la **separatriz**, y pasa por el equilibrio inestable $\theta = \pi$.
2. **Subir $b$ a 0.3.** Las curvas se enroscan hacia $(0,0)$: el reposo es ahora
   un **atractor**. Todas las condiciones iniciales terminan ahí.
3. **$b = 0.3$ y subir $A$.** La energía que se pierde por rozamiento se repone
   desde afuera. Aparece un ciclo al que el sistema tiende.
4. **$b = 0.2$, $A \approx 1.5$, $\Omega \approx 2$, t max = 200.** La trayectoria
   deja de cerrarse sobre sí misma. Eso ya es caos, en el mismo péndulo de siempre.

> **El punto pedagógico:** *disipación ⟹ atractor*. Sin disipación no hay
> atractores, sólo órbitas que conservan la energía. Este es exactamente el
> concepto que necesitamos para la segunda parte.

---
## 6. El péndulo magnético: sensibilidad a las condiciones iniciales

Un péndulo largo con un imán en la punta, oscilando sobre **tres imanes fijos**
en el plano. Es un experimento de escritorio (se consigue armado como juguete) y
**no tiene solución analítica**.

Con el péndulo largo y oscilaciones no muy grandes, el modelo es un punto $(x,y)$
en el plano sometido a:

- una fuerza restitutiva hacia el centro, $-k\,\vec{r}$;
- rozamiento, $-b\,\dot{\vec{r}}$;
- la atracción de cada imán,
  $\sum_i \dfrac{\vec{r}_i - \vec{r}}{\left(|\vec{r}_i - \vec{r}|^2 + d^2\right)^{3/2}}$

donde $d$ es la altura a la que pasa el imán del péndulo sobre la mesa (y de paso
evita que la fuerza se vuelva infinita).

Hay **tres atractores**: los tres imanes. Con rozamiento, toda trayectoria termina
detenida en uno de ellos. La pregunta interesante no es *"¿dónde termina?"* sino
**"¿en cuál de los tres, según dónde lo solté?"** — y resulta que esa pregunta no
siempre tiene respuesta útil.

Acá viene un detalle técnico que vale la pena señalar, porque es la diferencia entre
que el deslizador responda al instante o que haya que esperar varios segundos.

Las funciones de abajo trabajan con **números sueltos** (`x`, `y`, `vx`, `vy`) en vez de
arreglos de numpy. Para dos trayectorias esto resulta unas **20 veces más rápido**:
numpy tiene un costo fijo por operación que sólo se amortiza cuando se manejan miles de
datos a la vez. Con dos péndulos, la aritmética común de Python gana por lejos.

Además, la integración **se corta apenas los dos péndulos se detuvieron**, en vez de
llegar siempre hasta el final.

In [ ]:
# posiciones de los tres imanes (vértices de un triángulo equilátero)
ANG      = [np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3]
IMANES_X = [np.cos(a) for a in ANG]
IMANES_Y = [np.sin(a) for a in ANG]
COLORES  = ["tomato", "steelblue", "mediumseagreen"]

K_RES, D_ALT = 0.5, 0.25        # constante del resorte, altura del imán


def paso_iman(x, y, vx, vy, dt, k, b, d):
    "Un paso de RK4 para UN péndulo magnético, con aritmética escalar."

    def f(x, y, vx, vy):
        ax = -k*x - b*vx                    # resorte al centro + rozamiento
        ay = -k*y - b*vy
        for i in range(3):                  # atracción de cada imán
            dx = IMANES_X[i] - x
            dy = IMANES_Y[i] - y
            r3 = (dx*dx + dy*dy + d*d)**1.5
            ax += dx/r3
            ay += dy/r3
        return vx, vy, ax, ay

    a1, b1, c1, e1 = f(x, y, vx, vy)
    a2, b2, c2, e2 = f(x + dt/2*a1, y + dt/2*b1, vx + dt/2*c1, vy + dt/2*e1)
    a3, b3, c3, e3 = f(x + dt/2*a2, y + dt/2*b2, vx + dt/2*c2, vy + dt/2*e2)
    a4, b4, c4, e4 = f(x + dt*a3,   y + dt*b3,   vx + dt*c3,   vy + dt*e3)
    return (x  + dt/6*(a1 + 2*a2 + 2*a3 + a4),
            y  + dt/6*(b1 + 2*b2 + 2*b3 + b4),
            vx + dt/6*(c1 + 2*c2 + 2*c3 + c4),
            vy + dt/6*(e1 + 2*e2 + 2*e3 + e4))


def iman_mas_cercano(x, y):
    d2 = [(x - IMANES_X[i])**2 + (y - IMANES_Y[i])**2 for i in range(3)]
    return int(np.argmin(d2))


def soltar_dos(x0, y0, delta, b, dt=0.02, tmax=500.0,
               guardar_cada=5, revisar_cada=100):
    "Suelta dos péndulos separados por delta y devuelve sus trayectorias."
    est = [(x0, y0, 0.0, 0.0), (x0 + delta, y0, 0.0, 0.0)]
    XS  = [[x0], [x0 + delta]]
    YS  = [[y0], [y0]]

    for i in range(int(tmax/dt)):
        est = [paso_iman(*e, dt, K_RES, b, D_ALT) for e in est]

        if (i + 1) % guardar_cada == 0:
            for j in range(2):
                XS[j].append(est[j][0])
                YS[j].append(est[j][1])

        if (i + 1) % revisar_cada == 0:          # ¿ya se detuvieron los dos?
            quietos = True
            for e in est:
                v = (e[2]**2 + e[3]**2)**0.5
                cerca = min(((e[0]-IMANES_X[m])**2 + (e[1]-IMANES_Y[m])**2)**0.5
                            for m in range(3))
                if not (v < 1e-3 and cerca < 0.4):
                    quietos = False
                    break
            if quietos:
                break

    return XS, YS, [iman_mas_cercano(e[0], e[1]) for e in est]

### 6.1 Dos sueltas casi idénticas

Soltamos el péndulo desde un punto, y después desde otro punto separado del
primero por una distancia $\delta$ minúscula.

In [ ]:
@interact(x0=FloatSlider(min=-1.8, max=1.8, step=0.05, value=0.30,
                         description="x₀", continuous_update=False),
          y0=FloatSlider(min=-1.8, max=1.8, step=0.05, value=-1.40,
                         description="y₀", continuous_update=False),
          delta=SelectionSlider(options=[("1e-1",1e-1), ("1e-2",1e-2), ("1e-3",1e-3),
                                         ("1e-4",1e-4), ("1e-5",1e-5), ("1e-6",1e-6)],
                                value=1e-3, description="δ"),
          b=FloatSlider(min=0.05, max=0.30, step=0.05, value=0.10,
                        description="b (roce)", continuous_update=False))
def dos_sueltas(x0, y0, delta, b):
    XS, YS, destinos = soltar_dos(x0, y0, delta, b)

    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    ax.plot(XS[0], YS[0], lw=0.9, color="black",  alpha=0.75,
            label=f"suelta A  → imán {destinos[0]+1}")
    ax.plot(XS[1], YS[1], lw=0.9, color="orange", alpha=0.9,
            label=f"suelta B  → imán {destinos[1]+1}")
    for i in range(3):
        ax.plot(IMANES_X[i], IMANES_Y[i], "o", ms=17, color=COLORES[i],
                mec="k", mew=1.2)
        ax.annotate(str(i+1), (IMANES_X[i], IMANES_Y[i]),
                    ha="center", va="center", fontsize=9)
    ax.plot(x0, y0, "k*", ms=14, zorder=6)
    ax.set_aspect("equal"); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    igual = "MISMO imán" if destinos[0] == destinos[1] else "¡IMANES DISTINTOS!"
    ax.set_title(f"δ = {delta:.0e}   →   {igual}")
    ax.legend(loc="upper right", fontsize=8)
    plt.show()

✏️ **Para probar:** con el punto que viene por defecto — $(0.30,\,-1.40)$ con
$b = 0.10$ — bajen $\delta$ desde $10^{-1}$ hasta $10^{-6}$. Dos sueltas que
difieren en **una millonésima** terminan en imanes distintos.

Ningún experimentalista puede controlar la condición inicial con esa precisión.
El resultado es, en la práctica, **impredecible aunque el sistema sea
perfectamente determinista**. Ésta es la frase para dejar escrita en el pizarrón:

> **Determinista no es lo mismo que predecible.**

Prueben también dos variantes:

- **Cerca de un imán**, por ejemplo $(0,\,1)$: ahí el resultado es estable y
  $\delta$ no cambia nada. **No todo el plano es sensible.**
- **Subiendo el rozamiento a $b = 0.30$**, desde el mismo punto de partida: la
  sensibilidad desaparece. Con más rozamiento el péndulo se frena antes de poder
  "dudar".

O sea que la sensibilidad **no es una propiedad del sistema entero**, sino de la
región donde uno lo suelta y de cuánto alcanza a deambular antes de frenarse. Con
poco rozamiento el péndulo recorre mucho antes de decidirse, y por dónde pase
depende críticamente de dónde salió.

---
## 7. Para llevarse

| Sistema | ¿Solución cerrada? | ¿Predecible? |
|---|---|---|
| Péndulo linealizado | sí, elemental | totalmente |
| Péndulo exacto | sí, con funciones elípticas | totalmente |
| Péndulo forzado y disipativo | no | no siempre |
| Péndulo magnético | no | **no, en la práctica** |

Tres ideas para cerrar el día:

1. **La no linealidad, sola, no produce caos.** El péndulo exacto es no lineal y es
   perfectamente regular: lo resolvimos con una integral elíptica y su período se
   predice con toda precisión.

2. **Hace falta algo más.** O bien disipación combinada con inyección de energía (el
   péndulo forzado de la sección 5), o bien **varios atractores compitiendo** (el
   péndulo magnético). En ambos casos aparece algo que la ecuación sola no anticipaba.

3. **Determinista no es lo mismo que predecible.** No hay ninguna aleatoriedad en las
   ecuaciones del péndulo magnético: tres imanes, un resorte, rozamiento. Y sin embargo
   no podemos anticipar dónde termina, porque no podemos fijar la condición inicial con
   precisión infinita.

> **Para el aula:** el péndulo magnético es un objeto de escritorio y la demostración se
> hace en treinta segundos delante de la clase — se suelta dos veces "desde el mismo
> lugar" y termina en imanes distintos. La pregunta *"¿por qué no se puede repetir el
> experimento?"* abre todo el tema.

**Mañana:** el mapa logístico. Vamos a ver que no hace falta ni siquiera una ecuación
diferencial para tener este comportamiento — alcanza con una parábola y una calculadora
de bolsillo. Y vamos a encontrar un número universal escondido adentro.